In [0]:
from pyspark.sql.types import *
dbutils.widgets.text('catalog_name','zomato_analytics')
dbutils.widgets.text('database_name','audit')
dbutils.widgets.text('table_name','pipeline_audit_log')

In [0]:
from pyspark.sql.types import *
import datetime
class audit:
    def __init__(self, catalog_name=None, database_name=None, table_name=None):
        self.catalog_name = catalog_name or dbutils.widgets.get("catalog_name")
        self.database_name = database_name or dbutils.widgets.get("database_name")
        self.table_name = table_name or dbutils.widgets.get("table_name")
        
    def create_audit_catalog(self):
        spark.sql(f"create database if not exists {self.catalog_name}.{self.database_name}")
    
    def create_audit_table(self):
        spark.sql(f"""create table if not exists {self.catalog_name}.{self.database_name}.{self.table_name}
                  (
                    id Bigint  GENERATED ALWAYS AS IDENTITY,
                    run_datetime Timestamp,
                    workflow_name string,
                    task_name string,
                    layer string,
                    table_name string,
                    total_records BIGINT,
                    load_date Timestamp,
                    status  string,
                    error_message string
                  )""")
    def insert_audit_information(self,task_name,layer,table_name,total_records,status,error_message,workflow_name='manual',run_datetime= datetime.datetime.utcnow(),
                                 load_date = datetime.date.today()):
 

        schema = StructType([
            StructField("run_datetime", TimestampType(), True),
            StructField("workflow_name", StringType(), True),
            StructField("task_name", StringType(), True),
            StructField("layer", StringType(), True),
            StructField("table_name", StringType(), True),
            StructField("total_records", LongType(), True),
            StructField("load_date", TimestampType(), True),
            StructField("status", StringType(), True),
            StructField("error_message", StringType(), True),
        ])
        df=spark.createDataFrame([(run_datetime,workflow_name,task_name,layer,table_name,total_records,load_date,status,error_message)],schema=schema)
        df.write.mode('append').saveAsTable(f'{self.catalog_name}.{self.database_name}.{self.table_name}')